In [1]:
import torch
from vn_layers import VNLinear
from rotations import rot
torch.set_default_dtype(torch.float64) # NOTE For higher precision in the equivariance/invariance tests

In [45]:
latent_size = 100
fc1 = VNLinear(latent_size, latent_size)
fc2 = VNLinear(latent_size, latent_size)
torch.manual_seed(0)
rng_state = torch.get_rng_state()

def reparametrize(mu, logvar):
    
    # static rng so I cant test stuff
    g = torch.Generator()
    g.manual_seed(100)

    # rotation equivariance: For rotation equivariance, this should add relative to the vector direction. -> project into vector direction.
    std = logvar.mul(0.5).exp_()
    eps = std.new(std.size()).normal_(generator=g)

    return std+mu # for tests, we only return std+mu because eps will never be rotation equivariant.
    #return eps*std+mu 

def z_variational(z):
    mu = fc1(z)
    logvar = fc2(z) # TODO Logvar does no need to be 100x3 but only 100x1 -> this should be rotation invariant. -> do this with norm()?
    z = reparametrize(mu, logvar)
    return z, mu, logvar

In [27]:
z = torch.rand(5,100,3)
r = rot(*torch.rand(3))

Make sure the sampling is the same over multiple inputs -> that the static RNG works

In [23]:
z0,mu0,logvar0 = z_variational(z)
z1,mu1,logvar1 = z_variational(z)

assert torch.allclose(z0,z1,1e-8), "should now be the same after each sampling, -> static rng" 
assert torch.allclose(mu0, mu1,1e-8), "should now be the same after each sampling, -> static rng" 
assert torch.allclose(logvar0, logvar1,1e-8), "should now be the same after each sampling, -> static rng" 

after rotation, both z should still be the same.

In [28]:
z0,_,_ = z_variational(z@r)
z1,_,_ = z_variational(z)
z1 = z1@r

print("is rotation equivariant?",torch.allclose(z0,z1,1e-5))

is rotation equivariant? False


Now lets find out where the rotation equivariance gets lost 

In [29]:
_,mu0,logvar0 = z_variational(z@r)
_,mu1,logvar1 = z_variational(z)
mu1 = mu1@r
logvar1 = logvar1@r

assert torch.allclose(mu0, mu1, atol = 1e-6), "not equivariant to rotation"
assert torch.allclose(logvar0,logvar1,1e-6), "not rotation equivariant" 

mu and logvar are still rotation equivariant

In [43]:
_,_,logvar0 = z_variational(z@r)
_,_,logvar1 = z_variational(z)

std0 = logvar0.mul(0.5)
std1 = logvar1.mul(0.5)
std1 = std1@r

assert torch.allclose(std0, std1, atol = 1e-6), "not equivariant to rotation"

multiplying by 0.5 still keeps logvar rotation equivariant

In [44]:
_,_,logvar0 = z_variational(z@r)
_,_,logvar1 = z_variational(z)

std0 = logvar0.mul(0.5).exp_()
std1 = logvar1.mul(0.5).exp_()
std1 = std1@r

assert torch.allclose(std0, std1, atol = 1e-6), "not equivariant to rotation"

AssertionError: not equivariant to rotation

The exp is the issue!